# Full pipeline inference v2 -- prefix-KV-cache scoring (status.md "suggestion" section)

**New file, NOT an overwrite of `full_pipeline_infer.ipynb`** (v1, kernel `full-pipeline-infer`,
last at v11) -- that one stays as-is for rollback; this is a separate notebook/kernel.

Changes from v1:
- **Stage 3 (teacher-forced scoring) rewritten around a prefix KV cache**
  (`score_candidates_prefix_cache` in the helpers cell): forwards each row's context
  ONCE, expands the resulting cache across that row's candidates, then forwards only
  the short candidate continuations against the shared cache -- ~10x fewer
  token-forwards than v1's full-reprocess method (kept in this notebook too, unused,
  for reference/rollback). Verified numerically IDENTICAL to the old method on a real
  model (`scripts/score_candidates_prefix_cache.py --demo`, diff ~1e-6) before wiring
  in here, and the "tokenize candidate in isolation" boundary assumption it relies on
  was checked against the REAL Qwen/Mistral tokenizers on 300 real dev rows (0
  mismatches) before trusting it on Kaggle GPU-hours.
- `attn_implementation="sdpa"` set explicitly on both model loads -- confirms T4 gets
  the memory-efficient SDPA attention kernel rather than silently falling back to eager.
- Dev set stratified-subsampled to `DEV_SUBSAMPLE_SIZE` (20000 of 94488) for
  tuning+reporting when running the full set (not during the small smoke test) --
  ±0.6pt at 95% CI on the report number, cuts total rows 189k->~115k with zero effect
  on the test submission (test set is NEVER subsampled, all 94826 rows always run).

Same protocol as v1 otherwise: Qwen pinned `cuda:0`, Mistral pinned `cuda:1`, beam-k10
+ ngram-top10 candidates, `λ_qwen/λ_mistral` grid-tuned on a stratified first half of
dev and scored honestly on the second half, `test_set_pred.txt` output (one word/line,
no header, per the contest PDF's real spec).

**SMALL_BATCH_LIMIT below controls a smoke-test run first** -- confirms the new scoring
path fits + throughput before committing to the full run.

In [ ]:
# Kaggle's base image doesn't ship kenlm (local venv did, already installed) -- only
# the Python query bindings needed here (inference, not training), pip has those.
# torchao upgrade: Kaggle's preinstalled 0.10.0 is too old for peft's LoRA-loading
# torchao dispatcher check (wants >0.16.0) -- environment mismatch, not our code.
!pip install -q kenlm
!pip install -q -U torchao


In [ ]:
import os, glob, json, re, time, csv
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # Mistral fp16
# loads at 14.48/15.6GB on its T4 -- ~1.1GB nominal headroom, and repeated OOMs traced
# to allocator fragmentation from varying-shaped batches eating into that thin margin
# faster than the raw numbers suggest. Must be set before the first CUDA call.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, StoppingCriteria
from peft import PeftModel

import kenlm

print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i}", torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")


In [ ]:
# same content-based find as qwen3b_train_infer.ipynb -- don't guess mount paths
def find_file(name):
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert matches, f"{name} not found under /kaggle/input -- check the dataset is attached"
    return matches[0]

def find_by_config(model_type):
    for cfg_path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        try:
            cfg = json.load(open(cfg_path))
        except (json.JSONDecodeError, OSError):
            continue
        if cfg.get("model_type") == model_type:
            return os.path.dirname(cfg_path)
    return None

def find_adapter():
    matches = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    assert matches, "qwen LoRA adapter not found -- attach the qwen3b-lora-ckpt-6h dataset"
    return os.path.dirname(matches[0])

DEV_PATH = find_file("dev_set_final.csv")
TEST_PATH = find_file("test_set_no_answer_final.csv")
QWEN_BASE = find_by_config("qwen2")
QWEN_LORA = find_adapter()
MISTRAL_BASE = find_by_config("mistral") or "mistralai/Mistral-7B-v0.1"  # same weights
# uploaded once to krittiteen/mistral-7b-v0-1-base -- no more per-session HF hub
# redownload (~14GB every kernel start). Falls back to the hub id if that dataset
# isn't attached.
MISTRAL_LOCAL = MISTRAL_BASE != "mistralai/Mistral-7B-v0.1"
NGRAM_PATH = find_file("ngram_4_a.bin")
KN5_PATH = find_file("kn5.binary")
KN5_NUM_PATH = find_file("kn5_numbers.binary")

print("DEV_PATH:", DEV_PATH)
print("TEST_PATH:", TEST_PATH)
print("QWEN_BASE:", QWEN_BASE)
print("QWEN_LORA:", QWEN_LORA)
print("MISTRAL_BASE:", MISTRAL_BASE, "(local dataset)" if MISTRAL_LOCAL else "(HF hub download)")
print("NGRAM_PATH:", NGRAM_PATH)
print("KN5_PATH:", KN5_PATH)
print("KN5_NUM_PATH:", KN5_NUM_PATH)


## N-gram + KN5 models (CPU, kenlm) -- inlined from infer_ngram.py / blend_ngram.py / eval_number_kn5.py

In [ ]:
import math
LN10 = math.log(10)

def load_ngram_model(path):
    """Same format as infer_ngram.py's load_model -- pickled dict with keys
    n/counts/vocab/id_to_tok, NOT a plain tuple (unpacking the dict directly would
    silently bind the KEYS, not the values -- caught before running)."""
    import pickle
    with open(path, "rb") as f:
        m = pickle.load(f)
    return m["n"], m["counts"], m["vocab"], m["id_to_tok"]

def topk_by_letter(context_tokens, letter, n, counts, vocab, id_to_tok, k):
    """Verbatim from infer_ngram.py -- backoff-chain-merged top-k distinct words starting with letter."""
    ids = [vocab.get(t) for t in context_tokens[-(n - 1):]]
    seen, out = set(), []
    for j in range(len(ids), 0, -1):
        ctx_ids = ids[-j:]
        if None in ctx_ids:
            continue
        d = counts[j].get(tuple(ctx_ids))
        if not d:
            continue
        for wid, _ in sorted(d.items(), key=lambda kv: -kv[1]):
            w = id_to_tok[wid]
            if w and w[0] == letter and w not in seen:
                seen.add(w)
                out.append(w)
                if len(out) >= k:
                    return out
    uni = counts[0][()]
    for wid, _ in sorted(uni.items(), key=lambda kv: -kv[1]):
        w = id_to_tok[wid]
        if w and w[0] == letter and w not in seen:
            seen.add(w)
            out.append(w)
            if len(out) >= k:
                break
    return out


class KN5Scorer:
    """Verbatim from blend_ngram.py."""
    def __init__(self, path, order=5):
        self.model = kenlm.Model(path)
        self.order = order

    def score(self, context_tokens, candidate):
        ctx = context_tokens[-(self.order - 1):]
        text = " ".join(ctx + [candidate.lower()])
        log10p, _, _ = list(self.model.full_scores(text, bos=False, eos=False))[-1]
        return log10p * LN10


MAX_NUM_LEN = 8  # observed max anonymized digit-token length in train_final.src.tok

def pick_length(scorer, context_tokens):
    """Verbatim shape from eval_number_kn5.py."""
    best_len, best_score = 1, float("-inf")
    for length in range(1, MAX_NUM_LEN + 1):
        s = scorer.score(context_tokens, "1" * length)
        if s > best_score:
            best_len, best_score = length, s
    return best_len

FLOOR = math.log(1e-10)

n, counts, vocab, id_to_tok = load_ngram_model(NGRAM_PATH)
kn_scorer = KN5Scorer(KN5_PATH)
number_scorer = KN5Scorer(KN5_NUM_PATH)
print("n-gram + KN5 models loaded (CPU)")


## Qwen (cuda:0) + Mistral (cuda:1) -- fp16, no bnb-4bit

Falls back to both on cuda:0 (sequential use) if only one GPU is visible.

In [ ]:
QWEN_DEVICE = "cuda:0"
MISTRAL_DEVICE = "cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0"
print(f"Qwen -> {QWEN_DEVICE}, Mistral -> {MISTRAL_DEVICE}")

qwen_tok = AutoTokenizer.from_pretrained(QWEN_LORA)
if qwen_tok.pad_token is None:
    qwen_tok.pad_token = qwen_tok.eos_token
qwen_tok.padding_side = "left"

t0 = time.time()
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_BASE, torch_dtype=torch.float16, attn_implementation="sdpa").to(QWEN_DEVICE)
qwen_model = PeftModel.from_pretrained(qwen_model, QWEN_LORA)
qwen_model.eval()
print(f"Qwen loaded fp16/sdpa on {QWEN_DEVICE} in {time.time()-t0:.1f}s, "
      f"attn_implementation={qwen_model.config._attn_implementation}")

mistral_tok = AutoTokenizer.from_pretrained(MISTRAL_BASE)
if mistral_tok.pad_token is None:
    mistral_tok.pad_token = mistral_tok.eos_token

t0 = time.time()
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_BASE, torch_dtype=torch.float16, attn_implementation="sdpa").to(MISTRAL_DEVICE)
mistral_model.eval()
print(f"Mistral loaded fp16/sdpa on {MISTRAL_DEVICE} in {time.time()-t0:.1f}s, "
      f"attn_implementation={mistral_model.config._attn_implementation}")

for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated: {torch.cuda.memory_allocated(i)/1e9:.2f}GB / {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")


## Shared inference helpers -- inlined from infer_topk.py / score_candidates.py

In [ ]:
CAT_NUMBER = re.compile(r"[0-9]+")
CAT_WORD = re.compile(r"(?=.*[a-z])[a-z']+", re.IGNORECASE)

def categorize(tok):
    """Answer-based category -- dev reporting ONLY (test has no answer column)."""
    if CAT_NUMBER.fullmatch(tok):
        return "number"
    if CAT_WORD.fullmatch(tok):
        return "word"
    return "symbol"

def route(first_letter):
    """Faithful production routing -- first_letter's OWN character class, works
    identically whether or not the true answer/category is known (report.md: real
    serving-time routing uses this, not categorize(answer))."""
    if first_letter.isalpha():
        return "word"
    if first_letter.isdigit():
        return "number"
    return "symbol"

def detect_boundary(tokenizer):
    vocab = tokenizer.get_vocab()
    counts = {"▁": 0, "Ġ": 0}
    for piece in vocab:
        if piece[:1] in counts:
            counts[piece[:1]] += 1
    return max(counts, key=counts.get)

def build_letter_masks(tokenizer, boundary, device, vocab_size):
    vocab = tokenizer.get_vocab()
    letters = list("abcdefghijklmnopqrstuvwxyz")
    masks = {c: torch.zeros(vocab_size, dtype=torch.bool) for c in letters}
    for piece, idx in vocab.items():
        if len(piece) > 1 and piece[0] == boundary and piece[1].lower() in masks:
            masks[piece[1].lower()][idx] = True
    return {c: m.to(device) for c, m in masks.items()}

def get_boundary_ids(tokenizer, boundary):
    return [idx for piece, idx in tokenizer.get_vocab().items() if piece.startswith(boundary)]

class FirstTokenLetterMask(LogitsProcessor):
    def __init__(self, letter_mask, prompt_len):
        self.letter_mask = letter_mask
        self.prompt_len = prompt_len
    def __call__(self, input_ids, scores):
        if input_ids.shape[1] == self.prompt_len:
            scores = scores.masked_fill(~self.letter_mask, float("-inf"))
        return scores

class StopAtWordBoundary(StoppingCriteria):
    def __init__(self, prompt_len, boundary_ids_tensor, eos_id):
        self.prompt_len = prompt_len
        self.boundary_ids_tensor = boundary_ids_tensor
        self.eos_id = eos_id
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] <= self.prompt_len + 1:
            return torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        last = input_ids[:, -1]
        return torch.isin(last, self.boundary_ids_tensor) | (last == self.eos_id)

@torch.inference_mode()
def predict_topk_batch(model, tokenizer, contexts, letters, masks, boundary_ids_tensor, device, k, max_extra=4):
    """Beam-k, verbatim shape from infer_topk.py's predict_topk_batch (with_scores
    path dropped -- we use teacher-forced scoring instead, not generate()'s own
    sequence_scores, per the already-proven "teacher-forced beats beam scores" result)."""
    enc = tokenizer(contexts, return_tensors="pt", padding=True).to(device)
    letter_mask = torch.stack([masks[l.lower()] for l in letters])
    letter_mask = letter_mask.repeat_interleave(k, dim=0)
    prompt_len = enc["input_ids"].shape[1]
    eos_id = tokenizer.eos_token_id
    out = model.generate(
        **enc, max_new_tokens=max_extra + 1, num_beams=k, num_return_sequences=k, do_sample=False,
        early_stopping=True,
        logits_processor=[FirstTokenLetterMask(letter_mask, prompt_len)],
        stopping_criteria=[StopAtWordBoundary(prompt_len, boundary_ids_tensor, eos_id)],
        pad_token_id=eos_id,
    )
    generated = out[:, prompt_len:].tolist()
    boundary_set = set(boundary_ids_tensor.tolist())
    results = []
    for b in range(len(contexts)):
        ranked, seen = [], set()
        for beam in range(k):
            row = generated[b * k + beam]
            cut = next((i for i, tid in enumerate(row) if i > 0 and (tid in boundary_set or tid == eos_id)), len(row))
            w = tokenizer.decode(row[:cut]).strip().lower()
            if w and w not in seen:
                seen.add(w)
                ranked.append(w)
        results.append(ranked)
    return results

@torch.inference_mode()
def score_candidates_batch(model, tokenizer, context, candidates, device):
    """Teacher-forced full-word log-prob, verbatim from score_candidates.py. Kept for
    reference/rollback -- run_pipeline uses score_candidates_prefix_cache below."""
    ctx_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    prompt_len = len(ctx_ids)
    seqs, cand_lens = [], []
    for cand in candidates:
        full_ids = tokenizer(context + " " + cand, add_special_tokens=False)["input_ids"]
        seqs.append(full_ids)
        cand_lens.append(len(full_ids) - prompt_len)
    max_len = max(len(s) for s in seqs)
    pad_id = tokenizer.pad_token_id
    input_ids = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    attn = torch.zeros((len(seqs), max_len), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s)
        attn[i, :len(s)] = 1
    input_ids, attn = input_ids.to(device), attn.to(device)
    logits = model(input_ids=input_ids, attention_mask=attn, use_cache=False).logits
    logprobs = torch.log_softmax(logits[:, :-1], dim=-1)  # stay fp16 -- casting the full
    # (batch, seq, vocab) tensor to fp32 was itself the allocation that OOM'd on Mistral's
    # ~1GB headroom (vocab=32000 makes even one batch's fp32 copy hundreds of MB)
    targets = input_ids[:, 1:]
    token_lp = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1).float()
    start = prompt_len - 1
    return [token_lp[i, start:start+clen].sum().item() for i, clen in enumerate(cand_lens)]

MAX_FLAT_BATCH = 24  # unused by run_pipeline in this v2 notebook (kept for the old
# score_candidates_batch_multi below, rollback reference only)

@torch.inference_mode()
def score_candidates_batch_multi(model, tokenizer, contexts, candidate_lists, device):
    """Cross-row batched teacher-forced scoring, v1's approach -- full-reprocess per
    candidate, just batched across rows+candidates for GPU utilization. Kept in this
    notebook UNUSED, for reference/rollback -- run_pipeline uses
    score_candidates_prefix_cache below instead (v2's actual change, ~10x fewer
    token-forwards, not just a bigger batch of the same wasteful ones).

    contexts: list of context strings, one per row in this chunk.
    candidate_lists: list of candidate-word lists, parallel to contexts.
    Returns: list of score lists, parallel to candidate_lists.
    """
    flat_seqs, flat_cand_lens, row_spans = [], [], []
    idx = 0
    for ctx, cands in zip(contexts, candidate_lists):
        ctx_ids = tokenizer(ctx, add_special_tokens=False)["input_ids"]
        prompt_len = len(ctx_ids)
        start = idx
        for cand in cands:
            full_ids = tokenizer(ctx + " " + cand, add_special_tokens=False)["input_ids"]
            flat_seqs.append(full_ids)
            flat_cand_lens.append(len(full_ids) - prompt_len)
            idx += 1
        row_spans.append((start, idx, prompt_len))

    if not flat_seqs:
        return [[] for _ in contexts]

    pad_id = tokenizer.pad_token_id
    token_lp_chunks = []
    for b_start in range(0, len(flat_seqs), MAX_FLAT_BATCH):
        b_seqs = flat_seqs[b_start:b_start + MAX_FLAT_BATCH]
        max_len = max(len(s) for s in b_seqs)
        input_ids = torch.full((len(b_seqs), max_len), pad_id, dtype=torch.long)
        attn = torch.zeros((len(b_seqs), max_len), dtype=torch.long)
        for i, s in enumerate(b_seqs):
            input_ids[i, :len(s)] = torch.tensor(s)
            attn[i, :len(s)] = 1
        input_ids, attn = input_ids.to(device), attn.to(device)

        logits = model(input_ids=input_ids, attention_mask=attn, use_cache=False).logits
        logprobs = torch.log_softmax(logits[:, :-1], dim=-1)  # stay fp16, see above
        targets = input_ids[:, 1:]
        token_lp_chunks.append(logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1).float().cpu())
        del input_ids, attn, logits, logprobs, targets
        with torch.cuda.device(device):
            torch.cuda.empty_cache()

    max_w = max(t.shape[1] for t in token_lp_chunks)
    token_lp = torch.cat([torch.nn.functional.pad(t, (0, max_w - t.shape[1])) for t in token_lp_chunks], dim=0)

    results = []
    for start, end, prompt_len in row_spans:
        s = prompt_len - 1
        results.append([token_lp[i, s:s + flat_cand_lens[i]].sum().item() for i in range(start, end)])
    return results


def _expand_cache(past_key_values, n):
    """Repeat a just-built Cache along the batch dim so it can be reused for n
    candidates in one forward call. Tries the named Cache API method first; falls
    back to manual per-layer tensor repeat for versions where the method name or
    Cache internals differ -- Kaggle's preinstalled transformers version isn't
    guaranteed to match the local venv's (5.5.0, has batch_repeat_interleave)."""
    if hasattr(past_key_values, "batch_repeat_interleave"):
        past_key_values.batch_repeat_interleave(n)
        return past_key_values
    if hasattr(past_key_values, "key_cache"):
        for i in range(len(past_key_values.key_cache)):
            past_key_values.key_cache[i] = past_key_values.key_cache[i].repeat_interleave(n, dim=0)
            past_key_values.value_cache[i] = past_key_values.value_cache[i].repeat_interleave(n, dim=0)
        return past_key_values
    return tuple((k.repeat_interleave(n, dim=0), v.repeat_interleave(n, dim=0)) for k, v in past_key_values)


@torch.inference_mode()
def score_candidates_prefix_cache(model, tokenizer, context, candidates, device):
    """v2's real change -- teacher-forced full-word log-prob for every candidate of
    ONE row, sharing a single context forward pass via a prefix KV cache instead of
    reprocessing the full context per candidate. ~10x fewer token-forwards than
    score_candidates_batch_multi above for the typical ~18-candidate row. Verified
    numerically identical to the full-reprocess method (see
    scripts/score_candidates_prefix_cache.py --demo, diff ~1e-6 on a real model) and
    the tokenization-boundary assumption it relies on was checked against the real
    Qwen/Mistral tokenizers on 300 real dev rows (0 mismatches) before trusting it here.

    context: string. candidates: list of candidate words.
    Returns: list of float log-probs, parallel to candidates.
    """
    if not candidates:
        return []

    ctx_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    cand_id_lists = [tokenizer(" " + cand, add_special_tokens=False)["input_ids"] for cand in candidates]

    prefix_len = len(ctx_ids)
    n = len(candidates)
    pad_id = tokenizer.pad_token_id

    ctx_input = torch.tensor([ctx_ids], device=device)
    out1 = model(input_ids=ctx_input, use_cache=True)
    past = _expand_cache(out1.past_key_values, n)
    first_tok_logprob_row = torch.log_softmax(out1.logits[0, -1].float(), dim=-1)

    cand_lens = [len(ids) for ids in cand_id_lists]
    max_len = max(cand_lens)
    cont_ids = torch.full((n, max_len), pad_id, dtype=torch.long)
    cont_mask = torch.zeros((n, max_len), dtype=torch.long)
    for i, ids in enumerate(cand_id_lists):
        cont_ids[i, :len(ids)] = torch.tensor(ids)
        cont_mask[i, :len(ids)] = 1
    cont_ids, cont_mask = cont_ids.to(device), cont_mask.to(device)

    full_attn_mask = torch.cat([torch.ones(n, prefix_len, dtype=torch.long, device=device), cont_mask], dim=1)
    position_ids = torch.arange(prefix_len, prefix_len + max_len, device=device).unsqueeze(0).expand(n, -1)

    if max_len > 1:
        out2 = model(input_ids=cont_ids, attention_mask=full_attn_mask, past_key_values=past,
                      position_ids=position_ids, use_cache=False)
        cont_logprobs = torch.log_softmax(out2.logits[:, :-1], dim=-1)
    else:
        cont_logprobs = None

    results = []
    for i, ids in enumerate(cand_id_lists):
        lp = first_tok_logprob_row[ids[0]].float().item()
        for t in range(1, len(ids)):
            lp += cont_logprobs[i, t - 1, ids[t]].float().item()
        results.append(lp)
    return results


qwen_vocab_size = qwen_model.get_output_embeddings().weight.shape[0]
qwen_boundary = detect_boundary(qwen_tok)
qwen_masks = build_letter_masks(qwen_tok, qwen_boundary, QWEN_DEVICE, qwen_vocab_size)
qwen_boundary_ids = torch.tensor(get_boundary_ids(qwen_tok, qwen_boundary), device=QWEN_DEVICE)
print("boundary marker:", repr(qwen_boundary))


## SMALL_BATCH_LIMIT -- smoke test first, per user request

Set to a small number (e.g. 100) to confirm both models fit + throughput is sane before
the full ~189k-row run. Bump to None only after the smoke test looks right.

In [ ]:
SMALL_BATCH_LIMIT = 100  # <-- smoke test. Set to None for the full run.
BEAM_K = 10
NGRAM_TOPK = 10
GEN_BATCH_SIZE = 16   # was 4 under bnb-4bit+k10 locally -- fp16 frees VRAM, raise it
DEV_SUBSAMPLE_SIZE = 20000  # None to use all 94488 dev rows. status.md's "suggestion"
# section: a stratified 20k dev subset gives +-0.6pt at 95% CI on the report number,
# more than enough to tune (lam_qwen, lam_mistral) and report honestly -- cuts total
# rows 189k->~115k with zero effect on the test submission (test set is NEVER
# subsampled, see below). Only applied on the full run, not the small smoke test.

def load_rows(path, limit=None, has_answer=True):
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if limit:
        rows = rows[:limit]
    out = []
    for r in rows:
        d = {"context": r["context"], "first_letter": r["first letter"]}
        if has_answer:
            d["answer"] = r["answer"]
        out.append(d)
    return out

def stratified_subsample(rows, target_n, seed=123):
    """Stratified sample of target_n dev rows (proportional per categorize(answer)
    group), shuffled with seed -- see DEV_SUBSAMPLE_SIZE comment above. TEST_PATH is
    never subsampled; all rows there must be predicted for submission."""
    import random
    rng = random.Random(seed)
    by_cat = {}
    for i, r in enumerate(rows):
        by_cat.setdefault(categorize(r["answer"]), []).append(i)
    selected = []
    for cat, idx in by_cat.items():
        idx = idx[:]
        rng.shuffle(idx)
        take = round(target_n * len(idx) / len(rows))
        selected.extend(idx[:take])
    rng.shuffle(selected)
    return [rows[i] for i in selected]

dev_rows = load_rows(DEV_PATH, SMALL_BATCH_LIMIT, has_answer=True)
if SMALL_BATCH_LIMIT is None and DEV_SUBSAMPLE_SIZE:
    dev_rows_full_n = len(dev_rows)
    dev_rows = stratified_subsample(dev_rows, DEV_SUBSAMPLE_SIZE)
    print(f"dev subsampled: {dev_rows_full_n} -> {len(dev_rows)} rows "
          f"(stratified, DEV_SUBSAMPLE_SIZE={DEV_SUBSAMPLE_SIZE})")
test_rows = load_rows(TEST_PATH, SMALL_BATCH_LIMIT, has_answer=False)
print(f"{len(dev_rows)} dev rows, {len(test_rows)} test rows loaded (limit={SMALL_BATCH_LIMIT})")


## Per-row-set pipeline: route -> (word: beam-k10 + ngram-top10 -> teacher-forced Qwen+Mistral+KN5 triples) / (number: KN5-number length) / (symbol: first_letter itself)

In [ ]:
def run_pipeline(rows):
    """Returns list of dicts, one per input row, each carrying its route and either a
    (candidate -> (qwen, mistral, kn5) triple) dict for word rows, or a direct
    prediction for number/symbol rows -- same shape blend3.py's pick_best expects."""
    t0 = time.time()
    results = [None] * len(rows)
    word_idx = []
    for i, r in enumerate(rows):
        cat = route(r["first_letter"])
        if cat == "symbol":
            results[i] = {"route": "symbol", "pred": r["first_letter"]}
        elif cat == "number":
            ctx_tokens = r["context"].split()
            pred = "1" * pick_length(number_scorer, ctx_tokens)
            results[i] = {"route": "number", "pred": pred}
        else:
            word_idx.append(i)

    print(f"{len(word_idx)}/{len(rows)} rows routed to word pipeline", flush=True)

    # stage 1: beam-k10 candidate generation (Qwen, cuda:0)
    beam_cands = [None] * len(rows)
    for start in range(0, len(word_idx), GEN_BATCH_SIZE):
        batch_idx = word_idx[start:start + GEN_BATCH_SIZE]
        contexts = [rows[i]["context"] for i in batch_idx]
        letters = [rows[i]["first_letter"] for i in batch_idx]
        preds = predict_topk_batch(qwen_model, qwen_tok, contexts, letters, qwen_masks,
                                    qwen_boundary_ids, QWEN_DEVICE, BEAM_K)
        for i, p in zip(batch_idx, preds):
            beam_cands[i] = p
        done = start + len(batch_idx)
        if done % (GEN_BATCH_SIZE * 10) == 0 or done == len(word_idx):
            el = time.time() - t0
            print(f"[gen {done}/{len(word_idx)}] {el:.1f}s, {done/max(el,1e-9):.2f} rows/s", flush=True)

    # stage 2: ngram-top10 candidates (CPU) + union
    candidate_sets = {}
    for i in word_idx:
        ctx_tokens = rows[i]["context"].split()
        ngram_cands = topk_by_letter(ctx_tokens, rows[i]["first_letter"], n, counts, vocab, id_to_tok, NGRAM_TOPK)
        candidate_sets[i] = list(dict.fromkeys((beam_cands[i] or []) + ngram_cands))

    # stage 3: teacher-forced scoring via PREFIX KV-CACHE (score_candidates_prefix_cache)
    # -- v2's real change vs v1's score_candidates_batch_multi. Forwards each row's
    # context ONCE, expands the cache across that row's candidates, forwards only the
    # short candidate continuations -- ~10x fewer token-forwards. Per-row (not
    # cross-row) batching: the natural batch per forward call is the row's own
    # candidate count (<=20 by construction, BEAM_K=10 + NGRAM_TOPK=10 deduped union),
    # already a reasonable GPU batch on its own -- no MAX_FLAT_BATCH cap or cross-row
    # flattening needed. Qwen then Mistral sequentially, same reasoning as v1 for
    # staying off a thread pool (see v1's stage 3 comment / status.md's v6-v9 crash log).
    t1 = time.time()
    done = 0
    for i in word_idx:
        cands = candidate_sets[i]
        if not cands:
            results[i] = {"route": "word", "triples": {}}
            continue
        ctx = rows[i]["context"]
        q_scores = score_candidates_prefix_cache(qwen_model, qwen_tok, ctx, cands, QWEN_DEVICE)
        m_scores = score_candidates_prefix_cache(mistral_model, mistral_tok, ctx, cands, MISTRAL_DEVICE)
        ctx_tokens = ctx.split()
        triples = {w: (q_scores[k], m_scores[k], kn_scorer.score(ctx_tokens, w)) for k, w in enumerate(cands)}
        results[i] = {"route": "word", "triples": triples}
        done += 1
        if done % 200 == 0 or done == len(word_idx):
            el = time.time() - t1
            print(f"[tf-score {done}/{len(word_idx)}] {el:.1f}s, {done/max(el,1e-9):.2f} rows/s", flush=True)

    print(f"pipeline done: {len(rows)} rows in {time.time()-t0:.1f}s", flush=True)
    return results


In [ ]:
print("=== SMOKE TEST: dev ===")
dev_results = run_pipeline(dev_rows)
print("\n=== SMOKE TEST: test ===")
test_results = run_pipeline(test_rows)

for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} peak allocated: {torch.cuda.max_memory_allocated(i)/1e9:.2f}GB")


## Lambda: stratified 50/50 split of dev's word rows -- tune on first half, freeze, score on second half

In [ ]:
def pick_best(triples, lam_q, lam_m):
    lam_n = 1 - lam_q - lam_m
    best_word, best_score = None, float("-inf")
    for w, (q, m, k) in triples.items():
        s = lam_q * q + lam_m * m + lam_n * k
        if s > best_score:
            best_word, best_score = w, s
    return best_word

def build_grid(steps):
    return [(i / (steps - 1), j / (steps - 1)) for i in range(steps) for j in range(steps - i)]

def eval_word(idx_list, dev_rows, dev_results, lam_q, lam_m):
    correct = total = 0
    for i in idx_list:
        if dev_results[i]["route"] != "word":
            continue
        pred = pick_best(dev_results[i]["triples"], lam_q, lam_m)
        correct += (pred or "").strip().lower() == dev_rows[i]["answer"].strip().lower()
        total += 1
    return correct, total

def stratified_half_split(rows, seed=42):
    """k=2 stratified split by categorize(answer) -- same round-robin logic as
    blend_ngram.py's stratified_kfold, inlined for k=2 specifically."""
    import random
    rng = random.Random(seed)
    by_cat = {}
    for i, r in enumerate(rows):
        by_cat.setdefault(categorize(r["answer"]), []).append(i)
    halves = [[], []]
    for cat, idx in by_cat.items():
        idx = idx[:]
        rng.shuffle(idx)
        for j, i in enumerate(idx):
            halves[j % 2].append(i)
    return halves

tune_idx, report_idx = stratified_half_split(dev_rows)
print(f"tune half: {len(tune_idx)} rows, report half: {len(report_idx)} rows")

grid = build_grid(11)
best_lams, best_acc = (1.0, 0.0), -1
for lam_q, lam_m in grid:
    c, t = eval_word(tune_idx, dev_rows, dev_results, lam_q, lam_m)
    acc = c / t if t else 0
    if acc > best_acc:
        best_lams, best_acc = (lam_q, lam_m), acc
print(f"tuned on first half: lam_qwen={best_lams[0]:.2f} lam_mistral={best_lams[1]:.2f} (tune acc {best_acc*100:.2f}%)")

# frozen lambda applied to the REPORT half -- honest, held-out final accuracy
cat_correct, cat_total = {}, {}
for i in report_idx:
    r, res = dev_rows[i], dev_results[i]
    cat = categorize(r["answer"])
    cat_total[cat] = cat_total.get(cat, 0) + 1
    if res["route"] == "symbol":
        pred = res["pred"]
    elif res["route"] == "number":
        pred = res["pred"]
    else:
        pred = pick_best(res["triples"], *best_lams)
    ok = (pred or "").strip().lower() == r["answer"].strip().lower()
    cat_correct[cat] = cat_correct.get(cat, 0) + ok

print(f"\n--- FINAL dev accuracy (report half, frozen lambda, n={len(report_idx)}) ---")
tot_c = tot_n = 0
for cat in ("word", "symbol", "number"):
    c, t = cat_correct.get(cat, 0), cat_total.get(cat, 0)
    tot_c += c; tot_n += t
    if t:
        print(f"{cat:8s} {c}/{t}  {c/t*100:.2f}%")
if tot_n:
    print(f"{'overall':8s} {tot_c}/{tot_n}  {tot_c/tot_n*100:.2f}%")


## Apply frozen lambda to test set -- write test_set_pred.txt (one word per line, no header)

In [ ]:
test_preds = []
for i, (r, res) in enumerate(zip(test_rows, test_results)):
    if res["route"] == "symbol":
        pred = res["pred"]
    elif res["route"] == "number":
        pred = res["pred"]
    else:
        pred = pick_best(res["triples"], *best_lams) or ""
    test_preds.append(pred)

assert len(test_preds) == len(test_rows), (len(test_preds), len(test_rows))
with open("/kaggle/working/test_set_pred.txt", "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(p + "\n")
print(f"wrote {len(test_preds)} predictions -> test_set_pred.txt "
      f"(SMALL_BATCH_LIMIT={SMALL_BATCH_LIMIT} -- NOT the full submission until that's None)")
